Title
# Forecast 2 — Ridge vs Lasso (MAE Comparison) + Final Forecast

Setup & Paths

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge, Lasso

def find_project_root(start: Path) -> Path:
    """
    Find the project root by searching upwards for 'data' and 'reports' folders.
    Works even if notebook location changes (src/ or root/).
    """
    p = start.resolve()
    for _ in range(8):
        if (p / "data").exists() and (p / "reports").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not locate project root containing 'data' and 'reports' folders.")

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

CLEAN_FILE = DATA_DIR / "clean_expenses.csv"

OUT_COMPARE = REPORTS_DIR / "models_comparison.csv"
OUT_NEXT_WEEK = REPORTS_DIR / "forecast_next_week.csv"
OUT_REPORT = REPORTS_DIR / "forecast_report.txt"

ROOT, CLEAN_FILE


(WindowsPath('D:/courses/En + Ai + Github/Chat GPT course/Smart_Expenses_Project/project'),
 WindowsPath('D:/courses/En + Ai + Github/Chat GPT course/Smart_Expenses_Project/project/data/clean_expenses.csv'))

Load Data (clean_expenses.csv)

In [2]:
df = pd.read_csv(CLEAN_FILE)

# Basic checks
print("Rows:", len(df))
print("Columns:", list(df.columns))

# Ensure required columns exist
required_any = {"signed_amount", "amount"}
required_cols = {"date", "type", "category"}
missing_base = required_cols - set(df.columns)
if missing_base:
    raise ValueError(f"Missing required columns: {missing_base}")

if not (required_any & set(df.columns)):
    raise ValueError("clean_expenses.csv must include either 'signed_amount' or 'amount'")

df.head()


Rows: 1247
Columns: ['date', 'type', 'category', 'amount', 'payment_method', 'description', 'day_name', 'week', 'is_weekend', 'signed_amount']


,date,type,category,amount,payment_method,description,day_name,week,is_weekend,signed_amount
0,2025-01-01,income,Income,1200000,transfer,Monthly salary,Wednesday,1,False,1200000
1,2025-01-01,expense,Shopping,65931,online,Gifts,Wednesday,1,False,-65931
2,2025-01-01,expense,Bills,142976,transfer,Mobile recharge,Wednesday,1,False,-142976
3,2025-01-01,expense,Food,11814,cash,Snacks,Wednesday,1,False,-11814
4,2025-01-02,expense,Shopping,79996,cash,Personal items,Thursday,1,False,-79996


Build Daily Net Series

In [3]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

amt_col = "signed_amount" if "signed_amount" in df.columns else "amount"
df[amt_col] = pd.to_numeric(df[amt_col], errors="coerce").fillna(0)

daily = df.groupby(df["date"].dt.date)[amt_col].sum().reset_index()
daily.columns = ["date", "y"]
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").reset_index(drop=True)

daily.head(), daily.tail(), len(daily)


(        date       y
 0 2025-01-01  979279
 1 2025-01-02 -240571
 2 2025-01-03 -133972
 3 2025-01-04 -126168
 4 2025-01-05 -133846,
           date       y
 391 2026-01-27 -182941
 392 2026-01-28 -267275
 393 2026-01-29  -72492
 394 2026-01-30  -70178
 395 2026-01-31 -188949,
 396)

Feature Engineering (Lags + Rolling)

In [4]:
def make_daily_features(ts: pd.DataFrame, max_lag: int = 7) -> pd.DataFrame:
    df_ = ts.copy()
    df_["dow"] = df_["date"].dt.dayofweek  # 0=Mon
    df_["is_weekend"] = (df_["dow"] >= 5).astype(int)

    for k in range(1, max_lag + 1):
        df_[f"lag{k}"] = df_["y"].shift(k)

    df_["roll7_mean"] = df_["y"].shift(1).rolling(7).mean()
    df_["roll7_std"]  = df_["y"].shift(1).rolling(7).std()

    df_ = df_.dropna().reset_index(drop=True)
    return df_

feats = make_daily_features(daily, max_lag=7)

feature_cols = ["dow", "is_weekend"] + [f"lag{k}" for k in range(1, 8)] + ["roll7_mean", "roll7_std"]
X = feats[feature_cols]
y = feats["y"]

print("Daily points (raw):", len(daily))
print("Daily points (after features):", len(feats))
feats.head()


Daily points (raw): 396
Daily points (after features): 389


,date,y,dow,is_weekend,lag1,lag2,lag3,lag4,lag5,lag6,lag7,roll7_mean,roll7_std
0,2025-01-08,-112947,2,0,-125138.0,-213909.0,-133846.0,-126168.0,-133972.0,-240571.0,979279.0,810.714286,433984.515301
1,2025-01-09,-205887,3,0,-112947.0,-125138.0,-213909.0,-133846.0,-126168.0,-133972.0,-240571.0,-155221.571429,50288.284324
2,2025-01-10,-113748,4,0,-205887.0,-112947.0,-125138.0,-213909.0,-133846.0,-126168.0,-133972.0,-150266.714286,41400.635405
3,2025-01-11,-312087,5,1,-113748.0,-205887.0,-112947.0,-125138.0,-213909.0,-133846.0,-126168.0,-147377.571429,43385.374697
4,2025-01-12,-125574,6,1,-312087.0,-113748.0,-205887.0,-112947.0,-125138.0,-213909.0,-133846.0,-173937.428571,74201.428879


Ridge/Lasso Model Comparison (MAE)

In [5]:
models = {
    "Ridge(alpha=1.0)": Ridge(alpha=1.0),
    "Ridge(alpha=10.0)": Ridge(alpha=10.0),
    "Lasso(alpha=0.01)": Lasso(alpha=0.01, max_iter=50000),
    "Lasso(alpha=0.1)": Lasso(alpha=0.1, max_iter=50000),
}

def evaluate_models(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    n = len(X)
    # Use time series split if enough rows; else fallback to last-day holdout
    n_splits = min(3, n - 1) if n >= 6 else 0

    rows = []
    if n_splits >= 2:
        splitter = TimeSeriesSplit(n_splits=n_splits)
        for name, model in models.items():
            maes = []
            for tr, te in splitter.split(X):
                model.fit(X.iloc[tr], y.iloc[tr])
                pred = model.predict(X.iloc[te])
                maes.append(mean_absolute_error(y.iloc[te], pred))
            rows.append({
                "model": name,
                "mae_mean": float(np.mean(maes)),
                "mae_std": float(np.std(maes)),
                "method": f"TimeSeriesSplit(n_splits={n_splits})"
            })
    else:
        # holdout last day
        X_train, X_test = X.iloc[:-1], X.iloc[-1:]
        y_train, y_test = y.iloc[:-1], y.iloc[-1:]
        for name, model in models.items():
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            rows.append({
                "model": name,
                "mae_mean": float(mean_absolute_error(y_test, pred)),
                "mae_std": 0.0,
                "method": "Holdout(last_day)"
            })

    return pd.DataFrame(rows).sort_values("mae_mean").reset_index(drop=True)

cmp_df = evaluate_models(X, y)
cmp_df


c:\Users\LOQ\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.817e+11, tolerance: 1.285e+09
  model = cd_fast.enet_coordinate_descent(
c:\Users\LOQ\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.258e+12, tolerance: 1.808e+09
  model = cd_fast.enet_coordinate_descent(
c:\Users\LOQ\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of 

,model,mae_mean,mae_std,method
0,Ridge(alpha=10.0),128291.675566,12615.185678,TimeSeriesSplit(n_splits=3)
1,Ridge(alpha=1.0),128794.531937,13335.010587,TimeSeriesSplit(n_splits=3)
2,Lasso(alpha=0.1),128929.646895,13528.661026,TimeSeriesSplit(n_splits=3)
3,Lasso(alpha=0.01),128929.662341,13528.693065,TimeSeriesSplit(n_splits=3)


Save Comparison Table

In [6]:
cmp_df.to_csv(OUT_COMPARE, index=False, encoding="utf-8")
OUT_COMPARE


WindowsPath('D:/courses/En + Ai + Github/Chat GPT course/Smart_Expenses_Project/project/reports/models_comparison.csv')

Train Best Model + Forecast Next 7 Days

In [7]:
def build_model(model_name: str):
    if model_name.startswith("Ridge"):
        alpha = float(model_name.split("alpha=")[1].split(")")[0])
        return Ridge(alpha=alpha)
    if model_name.startswith("Lasso"):
        alpha = float(model_name.split("alpha=")[1].split(")")[0])
        return Lasso(alpha=alpha, max_iter=50000)
    raise ValueError("Unknown model name")

best_model_name = cmp_df.loc[0, "model"]
best_mae = float(cmp_df.loc[0, "mae_mean"])

best_model = build_model(best_model_name)
best_model.fit(X, y)

def forecast_next_7_days(model, ts: pd.DataFrame, max_lag: int = 7) -> pd.DataFrame:
    hist = ts.copy().sort_values("date").reset_index(drop=True)
    preds = []
    last_date = hist["date"].iloc[-1]

    for i in range(1, 8):
        next_date = last_date + timedelta(days=i)
        dow = next_date.weekday()
        is_weekend = 1 if dow >= 5 else 0

        y_series = hist["y"].tolist()
        feats_row = {"dow": dow, "is_weekend": is_weekend}

        for k in range(1, max_lag + 1):
            feats_row[f"lag{k}"] = y_series[-k]

        last7 = y_series[-7:]
        feats_row["roll7_mean"] = float(np.mean(last7))
        feats_row["roll7_std"] = float(np.std(last7, ddof=1)) if len(last7) >= 2 else 0.0

        X_next = pd.DataFrame([feats_row])
        y_pred = float(model.predict(X_next)[0])

        preds.append({"date": next_date.date(), "predicted_daily_net": y_pred})

        # append prediction for recursive forecasting
        hist = pd.concat([hist, pd.DataFrame([{"date": pd.to_datetime(next_date), "y": y_pred}])], ignore_index=True)

    out = pd.DataFrame(preds)
    out["predicted_week_total"] = out["predicted_daily_net"].sum()
    return out

next_week_df = forecast_next_7_days(best_model, daily, max_lag=7)
next_week_df


,date,predicted_daily_net,predicted_week_total
0,2026-02-01,-108953.849720,-659238.524075
1,2026-02-02,-81227.689524,-659238.524075
2,2026-02-03,-92285.445835,-659238.524075
3,2026-02-04,-97948.552947,-659238.524075
4,2026-02-05,-91973.277792,-659238.524075
5,2026-02-06,-96052.555617,-659238.524075
6,2026-02-07,-90797.152640,-659238.524075


Save Forecast & Write Report

In [8]:
next_week_df.to_csv(OUT_NEXT_WEEK, index=False, encoding="utf-8")

lines = []
lines.append("Forecast 2 Report")
lines.append("=================")
lines.append(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append("Mode: DAILY -> Next week = sum of next 7 daily predictions")
lines.append(f"Daily points (raw): {len(daily)}")
lines.append(f"Daily points (after features): {len(feats)}")
lines.append("")
lines.append("Model comparison (sorted by MAE):")
lines.append(cmp_df.to_string(index=False))
lines.append("")
lines.append(f"Best model: {best_model_name}")
lines.append(f"Best MAE: {best_mae:.4f}")
lines.append("")
lines.append(f"Next week predicted total: {next_week_df['predicted_week_total'].iloc[0]:.2f}")
lines.append(f"Saved: {OUT_COMPARE.name}")
lines.append(f"Saved: {OUT_NEXT_WEEK.name}")

OUT_REPORT.write_text("\n".join(lines), encoding="utf-8")

OUT_NEXT_WEEK, OUT_REPORT


(WindowsPath('D:/courses/En + Ai + Github/Chat GPT course/Smart_Expenses_Project/project/reports/forecast_next_week.csv'),
 WindowsPath('D:/courses/En + Ai + Github/Chat GPT course/Smart_Expenses_Project/project/reports/forecast_report.txt'))